# 10_main_study_generation_T4x2.ipynb

Authenticated content is discovered under any account, name, mount, archive extension, or nesting. No runtime path or identity edits are accepted. NON_EVIDENCE_RUNTIME_SMOKE or PLANNED_NOT_EXECUTED; paper_evidence=false.


In [ ]:
# Generated content-authenticated runtime identity. Nothing in this cell is editable.
import hashlib, json, os, pathlib, shutil, subprocess, sys

import platform
print({"status": "IMMEDIATE_KERNEL_RUNTIME_PROBE", "executable": sys.executable,
       "implementation": platform.python_implementation(),
       "python": platform.python_version(), "architecture": platform.machine(),
       "system": platform.system(), "libc": platform.libc_ver()})

STAGE = 'generation'
PROVIDER = 'main_study'
NOTEBOOK_NAME = '10_main_study_generation_T4x2.ipynb'
STUDY = 'main_study_cvpr'
EXPECTED_GPUS = 2
ALLOW_SINGLE_GPU_FALLBACK = True
USE_REAL_MODEL = False
MAX_ITEMS = None
ALLOW_FULL_RUN = True
INITIAL_BATCH_SIZE = 4
GLOBAL_SEED = 12013
SCHEMA_VERSION = "certvic.cvpr.output.v2"
SNAPSHOT_CONTRACT = "UNIFIED_SNAPSHOT"
PROMPT_TEMPLATE_ID = "certification_yes_no_v1"
PROMPT_TEMPLATE = "{prompt}\n"
PROMPT_TEMPLATE_HASH = hashlib.sha256(PROMPT_TEMPLATE.encode("utf-8")).hexdigest()
PARSER_VERSION = "certvic.parse.v2"
PRIMARY_PROVIDERS = ["qwen2_5_vl_7b", "internvl_8b", "llava_onevision_7b"]
CANONICAL_RETURN_ZIP = 'main_generation_return.zip'
LOCAL_DESTINATION = 'data/runtime/main_generation_return.zip'
WORKING_ROOT = os.environ.get("CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")
INPUT_ROOTS = [value for value in os.environ.get("CERTVIC_INPUT_ROOTS", "").split(os.pathsep)
               if value] or ["/kaggle/input", "/kaggle/working"]
OUTPUT_DIR = str(pathlib.Path(WORKING_ROOT) / "certvic_cvpr")
RUNTIME_CONFIG = str(pathlib.Path(WORKING_ROOT) / "certvic_cvpr_runtime.json")
PROVIDER_PERMISSION_EVENTS = str(
    pathlib.Path(WORKING_ROOT) / f"{PROVIDER}_provider_permission_events.jsonl"
)
GENERATION_ENGINE = "structured_texture_patch"
SEMANTIC_ENGINE = "deterministic_preliminary"
INPAINTING_SNAPSHOT = None
INPAINTING_MANIFEST = None
ALLOW_USE_PREINSTALLED_ENVIRONMENT = True
REQUIRE_EXACT_ENVIRONMENT = True
def shard_for(item_id, n):
    return int(hashlib.sha256(str(item_id).encode("utf-8")).hexdigest(), 16) % n
for key, value in {
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "DIFFUSERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "HF_HUB_DISABLE_TELEMETRY": "1", "PIP_NO_INDEX": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
}.items():
    os.environ[key] = value


In [ ]:
import hashlib, json, os, pathlib, shutil, stat, sys, zipfile

DISCOVERY_ERRORS = {
    "missing": "CERTVIC_DISCOVERY_01_REQUIRED_ROLE_NOT_FOUND",
    "ambiguous": "CERTVIC_DISCOVERY_02_AMBIGUOUS_DISTINCT_CONTENT",
    "authentication": "CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED",
}
DISCOVERY_POLICY = "CONTENT_AUTHENTICATED_ANY_LOCATION"
OPERATIONAL_FIELDS = {
    "builder_command", "created_time", "expected_kaggle_dataset_slug", "mount_path",
    "required_notebook", "validation_command",
}

def early_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def early_safe_member(info):
    name = info.filename
    normalized = name.replace("\\", "/")
    value = pathlib.PurePosixPath(normalized)
    mode = (info.external_attr >> 16) & 0xFFFF
    if (not normalized or normalized != name or normalized.endswith("/") or value.is_absolute()
            or ".." in value.parts or "." in value.parts or normalized.startswith("~")
            or "\x00" in normalized or info.is_dir() or stat.S_ISLNK(mode)
            or (mode and not stat.S_ISREG(mode))):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe member {name!r}")
    return value.as_posix()

def early_content_identity(manifest, hash_files):
    identity_manifest = {key: value for key, value in manifest.items()
                         if key not in OPERATIONAL_FIELDS}
    identity_files = {name: record for name, record in hash_files.items()
                      if name not in {"README.md", "bundle_manifest.json"}}
    payload = json.dumps({"manifest": identity_manifest, "files": identity_files},
                         indent=2, sort_keys=True).encode() + b"\n"
    return hashlib.sha256(payload).hexdigest()

def early_verify_archive(path):
    with zipfile.ZipFile(path) as archive:
        infos = archive.infolist()
        names = [early_safe_member(info) for info in infos]
        if "bundle_manifest.json" not in names or "hash_manifest.json" not in names:
            return None
        if len(names) != len(set(names)) or archive.testzip() is not None:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: duplicate or corrupt archive")
        manifest_bytes = archive.read("bundle_manifest.json")
        hash_bytes = archive.read("hash_manifest.json")
        if len(manifest_bytes) > 8 * 1024 * 1024 or len(hash_bytes) > 8 * 1024 * 1024:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: oversized manifest")
        manifest = json.loads(manifest_bytes)
        hashes = json.loads(hash_bytes)
        if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
                or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
                or manifest.get("bundle_type") != "CODE"):
            return None
        declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
        if (set(names) != set(hash_files) | {"hash_manifest.json"}
                or set(declared) != set(names) - {"bundle_manifest.json", "hash_manifest.json"}):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: file universe mismatch")
        for name, record in hash_files.items():
            payload = archive.read(name)
            observed = {"size": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}
            if record != observed or (name in declared and declared[name] != observed):
                raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: byte mismatch {name}")
        return manifest, hash_files, hashlib.sha256(manifest_bytes).hexdigest()

def early_verify_directory(path):
    root = pathlib.Path(path).resolve()
    manifest_path, hash_path = root / "bundle_manifest.json", root / "hash_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    hashes = json.loads(hash_path.read_text(encoding="utf-8"))
    if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
            or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
            or manifest.get("bundle_type") != "CODE"):
        return None
    observed = {}
    for member in root.rglob("*"):
        mode = member.lstat().st_mode
        if member.is_symlink() or not (stat.S_ISDIR(mode) or stat.S_ISREG(mode)):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe extracted member")
        if stat.S_ISREG(mode):
            observed[member.relative_to(root).as_posix()] = member
    declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
    if (set(observed) != set(hash_files) | {"hash_manifest.json"}
            or set(declared) != set(observed) - {"bundle_manifest.json", "hash_manifest.json"}):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted universe mismatch")
    for name, record in hash_files.items():
        member = observed[name]
        actual = {"size": member.stat().st_size, "sha256": early_sha256(member)}
        if actual != record or (name in declared and declared[name] != actual):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted byte mismatch {name}")
    return manifest, hash_files, early_sha256(manifest_path)

configured_roots = [value for value in os.environ.get("CERTVIC_INPUT_ROOTS", "").split(os.pathsep)
                    if value]
if not configured_roots:
    configured_roots = ["/kaggle/input", "/kaggle/working"]
INPUT_ROOTS = sorted({str(pathlib.Path(value).resolve()) for value in configured_roots
                      if pathlib.Path(value).is_dir() and not pathlib.Path(value).is_symlink()})
archive_candidates, directory_candidates = [], []
for root_value in INPUT_ROOTS:
    root = pathlib.Path(root_value)
    for current, directory_names, file_names in os.walk(root, followlinks=False):
        base = pathlib.Path(current)
        directory_names[:] = sorted(name for name in directory_names
                                     if not (base / name).is_symlink())
        if "bundle_manifest.json" in file_names and "hash_manifest.json" in file_names:
            directory_candidates.append(base.resolve())
        for name in sorted(file_names):
            candidate = base / name
            if candidate.is_symlink() or not candidate.is_file():
                continue
            try:
                with candidate.open("rb") as handle:
                    magic = handle.read(4)
            except OSError:
                continue
            if magic in {b"PK\x03\x04", b"PK\x05\x06", b"PK\x07\x08"}:
                archive_candidates.append(candidate.resolve())

valid, failures = [], []
for representation, candidates in (("zip_archive", sorted(set(archive_candidates))),
                                   ("extracted_directory", sorted(set(directory_candidates)))):
    for candidate in candidates:
        try:
            result = (early_verify_archive(candidate) if representation == "zip_archive"
                      else early_verify_directory(candidate))
        except (OSError, KeyError, json.JSONDecodeError, UnicodeDecodeError,
                zipfile.BadZipFile, RuntimeError) as error:
            failures.append(f"{candidate}: {error}")
            continue
        if result is None:
            continue
        manifest, hash_files, manifest_hash = result
        identity = early_content_identity(manifest, hash_files)
        expected = os.environ.get("CERTVIC_EXPECTED_CONTENT_ID_CODE")
        if expected and identity != expected.lower():
            failures.append(f"{candidate}: expected CODE content identity mismatch")
            continue
        valid.append({"path": candidate, "representation": representation,
                      "manifest": manifest, "manifest_sha256": manifest_hash,
                      "content_identity_sha256": identity})
if not valid:
    code = DISCOVERY_ERRORS["authentication"] if failures else DISCOVERY_ERRORS["missing"]
    raise RuntimeError(f"{code}: role=CODE failures={failures}")
identities = {row["content_identity_sha256"] for row in valid}
if len(identities) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['ambiguous']}: role=CODE candidates="
                       f"{[(row['content_identity_sha256'], str(row['path'])) for row in valid]}")
selected = min(valid, key=lambda row: os.path.normcase(str(row["path"])))
CODE_DISCOVERY_MIRRORS = sorted({str(row["path"]) for row in valid})
CODE_BUNDLE_SOURCE = str(selected["path"])
CODE_BUNDLE_HASH = selected["content_identity_sha256"]
CODE_ARCHIVE_SHA256 = (early_sha256(selected["path"])
                       if selected["representation"] == "zip_archive" else None)
if selected["representation"] == "zip_archive":
    CODE_EXTRACT_ROOT = pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_code"
    if CODE_EXTRACT_ROOT.exists():
        if CODE_EXTRACT_ROOT.is_symlink() or not CODE_EXTRACT_ROOT.is_dir():
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe CODE destination")
        shutil.rmtree(CODE_EXTRACT_ROOT)
    CODE_EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(selected["path"]) as archive:
        for info in archive.infolist():
            name = early_safe_member(info)
            output = (CODE_EXTRACT_ROOT / name).resolve()
            output.relative_to(CODE_EXTRACT_ROOT.resolve())
            output.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as reader, output.open("xb") as writer:
                shutil.copyfileobj(reader, writer, length=1024 * 1024)
else:
    CODE_EXTRACT_ROOT = pathlib.Path(selected["path"])
CODE_BUNDLE_PATH = (CODE_BUNDLE_SOURCE if selected["representation"] == "zip_archive"
                    else str(CODE_EXTRACT_ROOT / "bundle_manifest.json"))
CODE_BUNDLE = CODE_BUNDLE_PATH
project_candidates = sorted(path.parent.resolve() for path in CODE_EXTRACT_ROOT.rglob("pyproject.toml")
                            if (path.parent / "certvic/__init__.py").is_file())
if len(project_candidates) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: CODE project root ambiguous")
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
from certvic.cvpr.content_discovery import (
    DISCOVERY_POLICY, discover_authenticated_input, resolve_content_bound_roles,
)
from certvic.cvpr.notebook_bootstrap import discover_unique_file, discover_unique_root
authenticated_code = discover_authenticated_input(
    "CODE", roots=INPUT_ROOTS, expected_identity=CODE_BUNDLE_HASH,
    materialization_root=pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_authenticated_inputs",
)
if authenticated_code["content_identity_sha256"] != CODE_BUNDLE_HASH:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: early/shared CODE identity mismatch")
AUTHENTICATED_CONTENT_IDENTITIES = {"code_bundle": CODE_BUNDLE_HASH}
DISCOVERED_PROVENANCE = {"CODE": authenticated_code}
print({"discovery_policy": DISCOVERY_POLICY, "role": "CODE", "provider": None,
       "study": selected["manifest"].get("study"), "stage": selected["manifest"].get("stage"),
       "representation": selected["representation"], "discovered_path": CODE_BUNDLE_SOURCE,
       "content_identity_sha256": CODE_BUNDLE_HASH, "archive_sha256": CODE_ARCHIVE_SHA256,
       "mirrors": CODE_DISCOVERY_MIRRORS, "project_root": str(PROJECT_ROOT)})

from certvic.cvpr.environment_lock import (
    environment_lock_hash, load_environment_lock, select_locked_runtime,
)
from certvic.cvpr.runtime_profiles import discover_runtime_wheelhouse, runtime_probe

DISCOVERY_MATERIALIZATION_ROOT = pathlib.Path(WORKING_ROOT) / "certvic_authenticated_inputs"
CONFIG_DATASET = discover_authenticated_input(
    "CONFIGS", roots=INPUT_ROOTS,
    materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
)
TOOLS_DATASET = discover_authenticated_input(
    "EXECUTION_TOOLS", roots=INPUT_ROOTS,
    materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
)
for discovered in (CONFIG_DATASET, TOOLS_DATASET):
    DISCOVERED_PROVENANCE[discovered["role"]] = discovered
    print({key: discovered[key] for key in (
        "role", "provider", "study", "stage", "representation", "discovered_path",
        "materialized_root", "content_identity_sha256", "archive_sha256", "mirrors",
        "observed_mount", "observed_dataset_folder",
    )})
CONFIG_ROOT = pathlib.Path(CONFIG_DATASET["materialized_root"])
ENVIRONMENT_LOCK = str(discover_unique_file(CONFIG_ROOT, "kaggle_t4x2_environment.lock.json"))
ENVIRONMENT_LOCK_HASH = environment_lock_hash(ENVIRONMENT_LOCK)
BOOTSTRAP_ENVIRONMENT_LOCK_HASH = ENVIRONMENT_LOCK_HASH
KERNEL_RUNTIME_PROBE = runtime_probe()
RUNTIME_PROFILE = select_locked_runtime(ENVIRONMENT_LOCK, probe=KERNEL_RUNTIME_PROBE)
RUNTIME_PROFILE_ID = RUNTIME_PROFILE["profile_id"]
RUNTIME_PROFILE_HASH = RUNTIME_PROFILE["profile_hash"]
EXPECTED_WHEELHOUSE_CONTENT_ID = os.environ.get("CERTVIC_EXPECTED_CONTENT_ID_WHEELHOUSE")
WHEELHOUSE_DATASET = discover_runtime_wheelhouse(
    RUNTIME_PROFILE, roots=INPUT_ROOTS,
    materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
    expected_content_identity=EXPECTED_WHEELHOUSE_CONTENT_ID,
)
DISCOVERED_PROVENANCE[WHEELHOUSE_DATASET["role"]] = WHEELHOUSE_DATASET
print({key: WHEELHOUSE_DATASET[key] for key in (
    "role", "provider", "study", "stage", "representation", "discovered_path",
    "materialized_root", "content_identity_sha256", "archive_sha256", "mirrors",
    "observed_mount", "observed_dataset_folder",
)})
WHEELHOUSE_ROOT = pathlib.Path(WHEELHOUSE_DATASET["materialized_root"])
WHEELHOUSE_CONTENT_IDENTITY_SHA256 = WHEELHOUSE_DATASET["content_identity_sha256"]
WHEELHOUSE_MANIFEST = str(discover_unique_file(WHEELHOUSE_ROOT, "wheelhouse_manifest.json"))
WHEELHOUSE_PATH = str(WHEELHOUSE_ROOT / "wheels")
if not pathlib.Path(WHEELHOUSE_PATH).is_dir():
    raise RuntimeError("KAGGLE_BOOTSTRAP_04_WHEELHOUSE_INVALID: wheels directory missing")
MODEL_REGISTRY = str(discover_unique_file(CONFIG_ROOT, "certvic_immutable_model_registry.json"))
ATTACHED_INPUT_HASHES = {
    "code": CODE_BUNDLE_HASH,
    "configs": CONFIG_DATASET["content_identity_sha256"],
    "tools": TOOLS_DATASET["content_identity_sha256"],
    "wheelhouse": WHEELHOUSE_DATASET["content_identity_sha256"],
}
AUTHENTICATED_CONTENT_IDENTITIES.update({
    "configs": CONFIG_DATASET["content_identity_sha256"],
    "tools": TOOLS_DATASET["content_identity_sha256"],
    "wheelhouse": WHEELHOUSE_DATASET["content_identity_sha256"],
})
print({"environment_lock": ENVIRONMENT_LOCK, "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
       "kernel_runtime_probe": KERNEL_RUNTIME_PROBE,
       "runtime_profile_id": RUNTIME_PROFILE_ID,
       "runtime_profile_hash": RUNTIME_PROFILE_HASH,
       "wheelhouse_manifest": WHEELHOUSE_MANIFEST,
       "authenticated_content_identities": AUTHENTICATED_CONTENT_IDENTITIES})

GENERATION_DATASET = discover_authenticated_input(
    'GENERATION_INPUT', provider="controls", study='main_study_cvpr', stage="generation",
    roots=INPUT_ROOTS, materialization_root=DISCOVERY_MATERIALIZATION_ROOT,
)
DISCOVERED_PROVENANCE["GENERATION_INPUT"] = GENERATION_DATASET
print({key: GENERATION_DATASET[key] for key in (
    "role", "provider", "study", "stage", "representation", "discovered_path",
    "materialized_root", "content_identity_sha256", "archive_sha256", "mirrors",
    "observed_mount", "observed_dataset_folder",
)})
GENERATION_ROOT = pathlib.Path(GENERATION_DATASET["materialized_root"])
EDIT_PLAN = str(discover_unique_file(GENERATION_ROOT, "source_manifest.jsonl"))
TASK_MANIFEST = EDIT_PLAN
TASK_BUNDLE_ROOT = str(GENERATION_ROOT)
TASK_BUNDLE_MANIFEST = str(GENERATION_ROOT / "bundle_manifest.json")
TASK_BUNDLE_HASH = GENERATION_DATASET["content_identity_sha256"]
RUN_TAG = f"{STUDY}_generation_{TASK_BUNDLE_HASH[:12]}"
MODEL_ID = "controls/no-model"
PROCESSOR_ID = MODEL_ID
MODEL_COMMIT = "0" * 40
PROCESSOR_COMMIT = "0" * 40
MODEL_PATH = str(GENERATION_ROOT)
PROCESSOR_PATH = MODEL_PATH
SNAPSHOT_MANIFEST = TASK_BUNDLE_MANIFEST
SNAPSHOT_MANIFEST_HASH = GENERATION_DATASET["manifest_sha256"]
SNAPSHOT_ROOT_HASH = TASK_BUNDLE_HASH
EXPECTED_ARCHITECTURE = "NO_MODEL_GENERATION_CONTROLS"
FINAL_TASK_FREEZE = None
FINAL_REVIEW_LEDGER = None
DETECTABILITY_GATE = None
SMOKE_GATE_JSON = None
MATRIX_AUTHORIZATION = None
PROVIDER_PERMISSION = None
STUDY_CONFIG = None
ATTACHED_INPUT_HASHES["tasks"] = TASK_BUNDLE_HASH
AUTHENTICATED_CONTENT_IDENTITIES["tasks"] = TASK_BUNDLE_HASH


In [ ]:
import certvic
from certvic.cvpr.contracts import canonical_json_bytes, sha256_bytes
from certvic.cvpr.environment_lock import (
    offline_environment_flags, prepare_offline_environment,
)
from certvic.cvpr.notebook_bootstrap import configure_offline_environment
from certvic.cvpr.notebook_permission_binding import derive_permission_binding
from certvic.cvpr.reconcile_provider_permissions import (
    transition_provider_permission, verify_matrix_authorization, verify_provider_permission,
)
from certvic.cvpr.run_contract import build_run_contract
from certvic.cvpr.runtime_preflight import hardware_report
from certvic.cvpr.schema_contract import OUTPUT_SCHEMA
from certvic.cvpr.smoke_gate import require_scientific_run_gate
from certvic.cvpr.t4x2 import derive_seed_manifest, detect_topology, write_seed_manifest

PACKAGE_SOURCE_HASH = hashlib.sha256((PROJECT_ROOT / "certvic/__init__.py").read_bytes()).hexdigest()
print({"certvic_source": certvic.__file__, "package_source_hash": PACKAGE_SOURCE_HASH})
configure_offline_environment()
if environment_lock_hash(ENVIRONMENT_LOCK) != ENVIRONMENT_LOCK_HASH:
    raise RuntimeError("environment lock hash mismatch")
if (offline_environment_flags().get("HF_HUB_OFFLINE") != "1"
        or offline_environment_flags().get("PIP_NO_INDEX") != "1"):
    raise RuntimeError("offline environment flag contract is incomplete")
if SCHEMA_VERSION != OUTPUT_SCHEMA:
    raise RuntimeError(f"mixed output schema prohibited: {SCHEMA_VERSION} != {OUTPUT_SCHEMA}")
if any(len(str(identity)) != 64 for identity in ATTACHED_INPUT_HASHES.values()):
    raise RuntimeError("CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED: invalid content identity")
if any(record.get("discovery_policy") != "CONTENT_AUTHENTICATED_ANY_LOCATION"
       for record in DISCOVERED_PROVENANCE.values()):
    raise RuntimeError("CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED: discovery policy drift")

# Evaluation permission verification and nonce claim occur before hardware inspection,
# snapshot/model access, CUDA access, adapter creation, or scientific output creation.
if STAGE == "evaluation":
    matrix_authorization = verify_matrix_authorization(MATRIX_AUTHORIZATION)
    require_scientific_run_gate(SMOKE_GATE_JSON, PRIMARY_PROVIDERS)
    permission_binding = derive_permission_binding(globals())
    active_runtime_contract_input = {
        "study": STUDY, "runtime_class": "SCIENTIFIC_RUN", "provider": PROVIDER,
        "model_id": MODEL_ID, "processor_id": PROCESSOR_ID,
        "model_commit": MODEL_COMMIT, "processor_commit": PROCESSOR_COMMIT,
        "model_snapshot_manifest_hash": SNAPSHOT_MANIFEST_HASH,
        "processor_snapshot_manifest_hash": SNAPSHOT_MANIFEST_HASH,
        "snapshot_status": "LOCAL_SNAPSHOT_BYTES_VERIFIED",
        "snapshot_contract": SNAPSHOT_CONTRACT,
        "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
        "runtime_profile_id": RUNTIME_PROFILE_ID,
        "runtime_profile_hash": RUNTIME_PROFILE_HASH,
        "wheelhouse_content_identity_sha256": WHEELHOUSE_CONTENT_IDENTITY_SHA256,
        "prompt_template_id": PROMPT_TEMPLATE_ID,
        "prompt_template_hash": PROMPT_TEMPLATE_HASH,
        "parser_version": PARSER_VERSION, "output_schema": SCHEMA_VERSION,
        "run_tag": RUN_TAG, "code_bundle_hash": CODE_BUNDLE_HASH,
        "seed": GLOBAL_SEED,
        "generation_parameters": {"do_sample": False, "temperature": 0.0,
                                  "max_new_tokens": 16},
    }
    active_run_contract = build_run_contract(
        active_runtime_contract_input,
        task_manifest_sha256=sha256_bytes(canonical_json_bytes(active_tasks)), strict=True,
    )
    permission = verify_provider_permission(
        PROVIDER_PERMISSION, matrix=matrix_authorization,
        expected_provider=PROVIDER, expected_run_tag=RUN_TAG,
    )
    if (permission["active_input_hashes"] != permission_binding["input_hashes"]
            or permission["active_scalars"] != permission_binding["scalars"]
            or permission["task_bundle_hash"] != TASK_BUNDLE_HASH
            or permission["environment_hash"] != ENVIRONMENT_LOCK_HASH
            or permission.get("runtime_profile_id") != RUNTIME_PROFILE_ID
            or permission.get("runtime_profile_hash") != RUNTIME_PROFILE_HASH
            or permission.get("wheelhouse_content_identity_sha256")
            != WHEELHOUSE_CONTENT_IDENTITY_SHA256
            or permission["snapshot_hash"] != SNAPSHOT_MANIFEST_HASH
            or permission["snapshot_root_hash"] != SNAPSHOT_ROOT_HASH
            or permission["code_hash"] != CODE_BUNDLE_HASH
            or permission["prompt_template_hash"] != PROMPT_TEMPLATE_HASH
            or permission["run_contract_hash"] != active_run_contract["run_contract_hash"]
            or permission["parser_version"] != PARSER_VERSION):
        raise RuntimeError("provider permission differs from authenticated runtime identity")
    permission_claim = transition_provider_permission(
        permission, PROVIDER_PERMISSION_EVENTS, to_state="CLAIMED",
        actor=NOTEBOOK_NAME, detail={"binding_hash": permission_binding["binding_hash"]},
    )

if STAGE in {"evaluation", "snapshot_smoke", "real_model_smoke"}:
    if any(value in {None, ""} for value in
           [MODEL_COMMIT, PROCESSOR_COMMIT, MODEL_PATH, SNAPSHOT_MANIFEST,
            SNAPSHOT_MANIFEST_HASH, EXPECTED_ARCHITECTURE]):
        raise RuntimeError("snapshot contract is incomplete")
    if SNAPSHOT_CONTRACT != "UNIFIED_SNAPSHOT":
        raise RuntimeError("current notebooks require the frozen unified snapshot contract")
    if pathlib.Path(MODEL_PATH).resolve() != pathlib.Path(PROCESSOR_PATH).resolve():
        raise RuntimeError("unified snapshot requires identical model and processor roots")
    if hashlib.sha256(pathlib.Path(SNAPSHOT_MANIFEST).read_bytes()).hexdigest() != SNAPSHOT_MANIFEST_HASH:
        raise RuntimeError("snapshot manifest file hash mismatch")
    snapshot = verify_manifest(MODEL_PATH, SNAPSHOT_MANIFEST, expected_model_id=MODEL_ID,
        expected_model_commit=MODEL_COMMIT, expected_processor_commit=PROCESSOR_COMMIT,
        expected_architecture=EXPECTED_ARCHITECTURE)
    if not snapshot["passed"]: raise RuntimeError(snapshot["errors"])

environment_verification = prepare_offline_environment(
    ENVIRONMENT_LOCK, wheelhouse=WHEELHOUSE_PATH, wheelhouse_manifest=WHEELHOUSE_MANIFEST,
    allow_preinstalled=ALLOW_USE_PREINSTALLED_ENVIRONMENT,
    require_exact=REQUIRE_EXACT_ENVIRONMENT,
    require_cuda=STAGE in {"generation", "evaluation", "real_model_smoke"},
    selected_profile=RUNTIME_PROFILE,
    content_identities=AUTHENTICATED_CONTENT_IDENTITIES,
)
if environment_verification["status"] not in {
    "ISOLATED_OFFLINE_VENV_INSTALLED_AND_VERIFIED",
}:
    raise RuntimeError("exact offline environment was not established")

RUNTIME_PYTHON = environment_verification["python_executable"]
if environment_verification["runtime_profile_hash"] != RUNTIME_PROFILE_HASH:
    raise RuntimeError("CERTVIC_RUNTIME_02_WHEELHOUSE_ABI_MISMATCH: profile hash drift")
# Static compatibility marker: hardware = hardware_report()
hardware = hardware_report(python_executable=RUNTIME_PYTHON)
print(hardware)
gpu_stage = STAGE in {"generation", "evaluation", "real_model_smoke"}
if gpu_stage and not hardware["cuda_available"]:
    raise RuntimeError("CUDA is required for this notebook stage")
gpu_count = hardware["gpu_count"]
if gpu_stage and gpu_count < 2 and not (gpu_count == 1 and ALLOW_SINGLE_GPU_FALLBACK):
    raise RuntimeError(f"No allowed GPU topology: {gpu_count}")
GPU_IDS = list(range(min(gpu_count, EXPECTED_GPUS))) if gpu_stage else []
single_gpu_fallback = gpu_stage and len(GPU_IDS) == 1
if single_gpu_fallback: print("single_gpu_fallback: deterministic sequential shards")
T4_PLAN = detect_topology(
    device_names=[row["name"] for row in hardware.get("gpus", [])],
    allow_single_t4=ALLOW_SINGLE_GPU_FALLBACK,
) if gpu_stage else None
if T4_PLAN is not None: print(T4_PLAN.as_dict())


In [ ]:
if not pathlib.Path(EDIT_PLAN).is_file(): raise RuntimeError("EDIT_PLAN is missing")
if MAX_ITEMS is None and not ALLOW_FULL_RUN:
    raise RuntimeError("choose a bounded MAX_ITEMS or explicitly set ALLOW_FULL_RUN=True")
rows = [json.loads(line) for line in pathlib.Path(EDIT_PLAN).read_text().splitlines() if line]
# MAX_ITEMS is global across the study, never per shard.
bounded_rows = rows if MAX_ITEMS is None else rows[:MAX_ITEMS]
seed_manifests = [derive_seed_manifest(
    global_seed=GLOBAL_SEED, study=STUDY, provider=PROVIDER,
    gpu_id=(GPU_IDS[shard] if len(GPU_IDS) > 1 else 0), shard_id=shard,
    task_ids=[str(row.get("edit_id", row.get("item_id"))) for row in bounded_rows
              if shard_for(str(row.get("edit_id", row.get("item_id"))), max(1, len(GPU_IDS))) == shard],
    attempts=2,
) for shard in range(max(1, len(GPU_IDS)))]
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
write_seed_manifest(pathlib.Path(OUTPUT_DIR) / "seed_manifest.json", {
    "schema": "certvic.kaggle.seed_manifest.v1", "collision_check": "PASS",
    "manifests": seed_manifests, "prospective": True, "paper_evidence": False,
})
processes = []
for shard, gpu in enumerate(GPU_IDS):
    shard_rows = [row for row in bounded_rows if shard_for(str(row.get("edit_id", row.get("item_id"))), len(GPU_IDS)) == shard]
    shard_path = pathlib.Path(OUTPUT_DIR) / f"edit_plan_shard_{shard}.jsonl"
    shard_path.parent.mkdir(parents=True, exist_ok=True)
    shard_path.write_text("".join(json.dumps(row, sort_keys=True) + "\n" for row in shard_rows))
    env = dict(os.environ); env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    env["PYTHONPATH"] = str(PROJECT_ROOT)
    module = "certvic.cvpr.generation" if PROVIDER == "controls" else "certvic.cvpr.semantic_edits"
    command = [RUNTIME_PYTHON, "-m", module, "--task-manifest", str(shard_path),
               "--out-dir", str(pathlib.Path(OUTPUT_DIR) / f"generation_shard_{shard}"),
               "--seed", "12013", "--allow-full-run", "--resume"]
    if module.endswith("generation"): command += ["--engine", GENERATION_ENGINE]
    if SEMANTIC_ENGINE == "manifest_verified_offline_inpainting":
        if module == "certvic.cvpr.generation":
            raise RuntimeError("specificity controls require deterministic engines; optional inpainting is a separate diagnostic")
        if not INPAINTING_SNAPSHOT or not INPAINTING_MANIFEST:
            raise RuntimeError("optional inpainting requires an explicit local snapshot and manifest")
        command += ["--inpainting-snapshot", INPAINTING_SNAPSHOT,
                    "--inpainting-manifest", INPAINTING_MANIFEST,
                    "--inpainting-model-id", MODEL_ID,
                    "--inpainting-model-commit", MODEL_COMMIT,
                    "--inpainting-architecture", EXPECTED_ARCHITECTURE,
                    "--batch-size", str(INITIAL_BATCH_SIZE)]
    log = open(pathlib.Path(OUTPUT_DIR) / f"generation_{shard}.stdout.log", "w")
    error = open(pathlib.Path(OUTPUT_DIR) / f"generation_{shard}.stderr.log", "w")
    processes.append((subprocess.Popen(command, env=env, stdout=log, stderr=error), log, error, shard))
for process, log, error, shard in processes:
    code = process.wait(); log.close(); error.close()
    if code: raise RuntimeError(f"generation shard {shard} failed; preserve outputs and inspect logs")


In [ ]:
required_outputs = ["merged_raw.jsonl", "runtime_manifest.json",
                    "environment_manifest.json", "validation_report.json",
                    "failure_report.json", "hash_manifest.json"]
if STAGE in {"evaluation", "mock_smoke", "real_model_smoke"}:
    expected_shards = 1 if STAGE in {"mock_smoke", "real_model_smoke"} else len(GPU_IDS)
    subprocess.run([RUNTIME_PYTHON, "-m", "certvic.cvpr.package_run",
                    "--frozen-runtime-config", RUNTIME_CONFIG,
                    "--expected-shards", str(expected_shards)], check=True,
                   env={**os.environ, "PYTHONPATH": str(PROJECT_ROOT)})
    package_source = pathlib.Path(OUTPUT_DIR) / f"certvic_cvpr_{RUN_TAG}_{PROVIDER}.zip"
    canonical_return = pathlib.Path(OUTPUT_DIR) / CANONICAL_RETURN_ZIP
    if STAGE == "evaluation":
        if not package_source.is_file(): raise RuntimeError("scientific package source ZIP is missing")
        if package_source != canonical_return: shutil.copyfile(package_source, canonical_return)
    if STAGE == "real_model_smoke":
        smoke_path = pathlib.Path(OUTPUT_DIR) / f"00C2_{PROVIDER}_real_model_smoke.zip"
        if not smoke_path.is_file():
            raise RuntimeError("package_run did not atomically create the canonical 00C2 ZIP")
        print({"canonical_smoke_zip": str(smoke_path)})
elif STAGE == "generation":
    root = pathlib.Path(OUTPUT_DIR); root.mkdir(parents=True, exist_ok=True)
    task_manifest_hash = hashlib.sha256(pathlib.Path(EDIT_PLAN).read_bytes()).hexdigest()
    generation_contract = {
        "schema": "certvic.cvpr.generation_run_contract.v1", "study": STUDY,
        "provider": PROVIDER, "task_manifest_sha256": task_manifest_hash,
        "code_bundle_hash": CODE_BUNDLE_HASH, "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
        "runtime_profile_id": RUNTIME_PROFILE_ID, "runtime_profile_hash": RUNTIME_PROFILE_HASH,
        "seed": 12013, "generation_engine": GENERATION_ENGINE,
        "semantic_engine": SEMANTIC_ENGINE, "paper_evidence": False,
    }
    generation_contract["run_contract_hash"] = hashlib.sha256(json.dumps(
        generation_contract, sort_keys=True, separators=(",", ":")
    ).encode()).hexdigest()
    generation_environment = {**hardware, "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
                              "runtime_profile_id": RUNTIME_PROFILE_ID,
                              "runtime_profile_hash": RUNTIME_PROFILE_HASH,
                              "runtime_python": RUNTIME_PYTHON,
                              "wheelhouse_content_identity_sha256": WHEELHOUSE_DATASET["content_identity_sha256"],
                              "offline_environment_status": environment_verification["status"],
                              "paper_evidence": False}
    generation_runtime = {
        "schema": "certvic.cvpr.generation_runtime.v1", "study": STUDY,
        "provider": PROVIDER, "run_contract_hash": generation_contract["run_contract_hash"],
        "code_bundle_hash": CODE_BUNDLE_HASH, "task_manifest_sha256": task_manifest_hash,
        "runtime_profile_id": RUNTIME_PROFILE_ID, "runtime_profile_hash": RUNTIME_PROFILE_HASH,
        "runtime_python": RUNTIME_PYTHON,
        "paper_evidence": False,
    }
    run_contract_path = root / "run_contract.json"
    environment_path = root / "environment_manifest.json"
    runtime_path = root / "runtime_manifest.json"
    run_contract_path.write_text(json.dumps(generation_contract, indent=2, sort_keys=True))
    environment_path.write_text(json.dumps(generation_environment, indent=2, sort_keys=True))
    runtime_path.write_text(json.dumps(generation_runtime, indent=2, sort_keys=True))
    generation_zip = root / CANONICAL_RETURN_ZIP
    subprocess.run([RUNTIME_PYTHON, "-m", "certvic.cvpr.package_generation",
                    "--study-manifest", EDIT_PLAN, "--generation-root", OUTPUT_DIR,
                    "--out-zip", str(generation_zip), "--assemble-shards",
                    "--run-contract", str(run_contract_path),
                    "--environment-manifest", str(environment_path),
                    "--runtime-manifest", str(runtime_path), "--strict"], check=True,
                   env={**os.environ, "PYTHONPATH": str(PROJECT_ROOT)})
elif STAGE in {"code_smoke", "snapshot_smoke"}:
    from certvic.cvpr.smoke_artifacts import (
        write_environment_artifacts, write_snapshot_artifacts,
    )
    out = pathlib.Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
    if STAGE == "code_smoke":
        canonical_artifacts = write_environment_artifacts(out, {
            "status": environment_verification["status"], "passed": True,
            "environment_hash": ENVIRONMENT_LOCK_HASH,
            "environment_lock_hash": ENVIRONMENT_LOCK_HASH,
            "code_bundle_hash": CODE_BUNDLE_HASH, "hardware": hardware,
        })
    else:
        canonical_artifacts = write_snapshot_artifacts(out, PROVIDER, {
            **snapshot, "snapshot_contract": SNAPSHOT_CONTRACT,
            "model_id": MODEL_ID, "model_commit": MODEL_COMMIT,
            "processor_commit": PROCESSOR_COMMIT,
            "snapshot_root_hash": SNAPSHOT_ROOT_HASH,
        })
    print(canonical_artifacts)
canonical_return_path = pathlib.Path(OUTPUT_DIR) / CANONICAL_RETURN_ZIP
if STAGE != "mock_smoke" and not canonical_return_path.is_file():
    raise RuntimeError(f"canonical return ZIP missing: {CANONICAL_RETURN_ZIP}")
if canonical_return_path.is_file():
    print({"canonical_return_zip": str(canonical_return_path),
           "sha256": hashlib.sha256(canonical_return_path.read_bytes()).hexdigest()})
print({"required_outputs": required_outputs, "paper_evidence": False})
if STAGE in {"code_smoke", "snapshot_smoke", "real_model_smoke"}:
    print({"local_handoff_command": "python3 -m certvic.cvpr.smoke_handoff --artifacts-dir <RETURNED_ARTIFACTS> --smoke-contract <TRUSTED_SMOKE_CONTRACT> --model-registry configs/models/certvic_cvpr_model_registry.yaml --environment-lock configs/runtime/kaggle_t4x2_environment.lock.json --out-dir <SMOKE_GATE_DIR>"})
elif STAGE == "evaluation":
    print({"local_import_command": "python3 -m certvic.cvpr.import_transaction run --matrix <MATRIX_AUTHORIZATION> --provider-zip qwen2_5_vl_7b=<QWEN_ZIP> --provider-zip internvl_8b=<INTERNVL_ZIP> --provider-zip llava_onevision_7b=<LLAVA_ZIP> --destination <CANONICAL_DESTINATION> --nonce-ledger <CONSUMED_NONCES>"})
